# TheLook E-Commerce — Data Analysis

**Pipeline**: `bigquery-public-data.thelook_ecommerce` → `ntupace.thelook_raw` → dbt ELT → `ntupace.thelook_marts`

**Star schema**: `fact_sales` joined to `dim_customer`, `dim_product`, `dim_date`

Analyses:
1. Monthly sales trends
2. Top-selling products
3. Revenue by category & price tier
4. Customer segmentation
5. Customer Lifetime Value (CLV)
6. Key business KPIs

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
from sqlalchemy import create_engine
import warnings
warnings.filterwarnings('ignore')

sns.set_theme(style='whitegrid')
plt.rcParams['figure.figsize'] = (13, 5)

PROJECT = 'ntupace'
DATASET = 'thelook_marts'
engine  = create_engine(f'bigquery://{PROJECT}/{DATASET}')

def q(sql):
    return pd.read_sql(sql, engine)

print('Connected to BigQuery:', PROJECT, '/', DATASET)

## 1. Monthly Sales Trends

In [ ]:
monthly = q("""
SELECT
    FORMAT_DATE('%Y-%m', created_at)   AS sale_month,
    COUNT(order_item_id)               AS total_items,
    COUNT(DISTINCT order_id)           AS total_orders,
    ROUND(SUM(sale_price), 2)          AS revenue,
    ROUND(SUM(gross_margin), 2)        AS profit,
    ROUND(AVG(sale_price), 2)          AS avg_price
FROM `ntupace.thelook_marts.fact_sales`
WHERE status NOT IN ('cancelled','returned')
  AND created_at IS NOT NULL
GROUP BY sale_month
ORDER BY sale_month
""")

fig, axes = plt.subplots(1, 2, figsize=(16, 5))

axes[0].plot(monthly['sale_month'], monthly['revenue'], marker='o', color='steelblue', linewidth=2)
axes[0].fill_between(range(len(monthly)), monthly['revenue'], alpha=0.1, color='steelblue')
axes[0].set_title('Monthly Revenue', fontsize=13, fontweight='bold')
axes[0].set_xlabel('Month')
axes[0].set_ylabel('Revenue ($)')
axes[0].set_xticks(range(0, len(monthly), max(1, len(monthly)//12)))
axes[0].set_xticklabels(
    monthly['sale_month'].iloc[::max(1, len(monthly)//12)],
    rotation=45, ha='right'
)
axes[0].yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'${x:,.0f}'))

axes[1].bar(range(len(monthly)), monthly['total_orders'], color='coral')
axes[1].set_title('Monthly Order Volume', fontsize=13, fontweight='bold')
axes[1].set_xlabel('Month')
axes[1].set_ylabel('Orders')
axes[1].set_xticks(range(0, len(monthly), max(1, len(monthly)//12)))
axes[1].set_xticklabels(
    monthly['sale_month'].iloc[::max(1, len(monthly)//12)],
    rotation=45, ha='right'
)

plt.tight_layout()
plt.savefig('../reports/monthly_trends.png', dpi=150, bbox_inches='tight')
plt.show()
monthly.tail(6)

## 2. Top-Selling Products

In [ ]:
top_products = q("""
SELECT
    p.name               AS product_name,
    p.category,
    p.brand,
    p.price_tier,
    COUNT(f.order_item_id)         AS units_sold,
    ROUND(SUM(f.sale_price), 2)    AS total_revenue,
    ROUND(SUM(f.gross_margin), 2)  AS total_profit
FROM `ntupace.thelook_marts.fact_sales`   f
JOIN `ntupace.thelook_marts.dim_product`  p ON f.product_key = p.product_key
WHERE f.status NOT IN ('cancelled','returned')
GROUP BY p.name, p.category, p.brand, p.price_tier
ORDER BY total_revenue DESC
LIMIT 15
""")

fig, ax = plt.subplots(figsize=(14, 7))
palette = {'Budget':'#4CAF50','Mid-range':'#2196F3','Premium':'#FF9800','Luxury':'#9C27B0'}
colors = [palette.get(t, '#607D8B') for t in top_products['price_tier']]
bars = ax.barh(top_products['product_name'], top_products['total_revenue'], color=colors)
ax.set_title('Top 15 Products by Revenue', fontsize=13, fontweight='bold')
ax.set_xlabel('Revenue ($)')
ax.invert_yaxis()
ax.xaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'${x:,.0f}'))

from matplotlib.patches import Patch
legend = [Patch(color=v, label=k) for k, v in palette.items()]
ax.legend(handles=legend, title='Price Tier', loc='lower right')

plt.tight_layout()
plt.savefig('../reports/top_products.png', dpi=150, bbox_inches='tight')
plt.show()
top_products

## 3. Revenue by Category & Price Tier

In [ ]:
by_category = q("""
SELECT
    p.category,
    p.department,
    p.price_tier,
    COUNT(f.order_item_id)              AS units_sold,
    ROUND(SUM(f.sale_price), 2)         AS revenue,
    ROUND(SUM(f.gross_margin), 2)       AS profit,
    ROUND(AVG(f.gross_margin_pct)*100, 1) AS avg_margin_pct
FROM `ntupace.thelook_marts.fact_sales`   f
JOIN `ntupace.thelook_marts.dim_product`  p ON f.product_key = p.product_key
WHERE f.status NOT IN ('cancelled','returned')
GROUP BY p.category, p.department, p.price_tier
ORDER BY revenue DESC
""")

cat_agg = by_category.groupby('category')[['revenue','profit']].sum().reset_index().sort_values('revenue', ascending=False).head(12)

fig, axes = plt.subplots(1, 2, figsize=(16, 6))

sns.barplot(data=cat_agg, x='revenue', y='category', palette='Blues_r', ax=axes[0])
axes[0].set_title('Top 12 Categories by Revenue', fontsize=13, fontweight='bold')
axes[0].set_xlabel('Revenue ($)')
axes[0].set_ylabel('')
axes[0].xaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'${x:,.0f}'))

tier_agg = by_category.groupby('price_tier')['revenue'].sum().reset_index()
axes[1].pie(tier_agg['revenue'], labels=tier_agg['price_tier'],
            autopct='%1.1f%%', startangle=140,
            colors=['#4CAF50','#2196F3','#FF9800','#9C27B0'])
axes[1].set_title('Revenue Share by Price Tier', fontsize=13, fontweight='bold')

plt.tight_layout()
plt.savefig('../reports/category_revenue.png', dpi=150, bbox_inches='tight')
plt.show()
by_category.head(10)

## 4. Customer Segmentation

In [ ]:
seg = q("""
SELECT
    c.age_bucket,
    c.gender,
    c.traffic_source,
    c.country,
    COUNT(DISTINCT f.customer_key)       AS customers,
    ROUND(SUM(f.sale_price), 2)          AS revenue,
    ROUND(AVG(f.sale_price), 2)          AS avg_order_value,
    COUNT(f.order_item_id)               AS total_items
FROM `ntupace.thelook_marts.fact_sales`    f
JOIN `ntupace.thelook_marts.dim_customer`  c ON f.customer_key = c.customer_key
WHERE f.status NOT IN ('cancelled','returned')
GROUP BY c.age_bucket, c.gender, c.traffic_source, c.country
ORDER BY revenue DESC
""")

age_rev = seg.groupby('age_bucket')['revenue'].sum().reset_index().sort_values('age_bucket')
src_rev = seg.groupby('traffic_source')['revenue'].sum().reset_index().sort_values('revenue', ascending=False)

fig, axes = plt.subplots(1, 2, figsize=(16, 5))

axes[0].pie(age_rev['revenue'], labels=age_rev['age_bucket'],
            autopct='%1.1f%%', startangle=140)
axes[0].set_title('Revenue Share by Age Group', fontsize=13, fontweight='bold')

sns.barplot(data=src_rev, x='revenue', y='traffic_source', palette='Purples_r', ax=axes[1])
axes[1].set_title('Revenue by Acquisition Channel', fontsize=13, fontweight='bold')
axes[1].set_xlabel('Revenue ($)')
axes[1].set_ylabel('')
axes[1].xaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'${x:,.0f}'))

plt.tight_layout()
plt.savefig('../reports/customer_segments.png', dpi=150, bbox_inches='tight')
plt.show()

print('Top 10 countries by revenue:')
seg.groupby('country')['revenue'].sum().sort_values(ascending=False).head(10)

## 5. Customer Lifetime Value (CLV)

In [ ]:
clv = q("""
SELECT
    c.customer_key,
    CONCAT(c.first_name, ' ', c.last_name)  AS full_name,
    c.age_bucket,
    c.gender,
    c.country,
    c.traffic_source,
    COUNT(DISTINCT f.order_id)              AS total_orders,
    ROUND(SUM(f.sale_price), 2)             AS lifetime_value,
    ROUND(AVG(f.sale_price), 2)             AS avg_order_value,
    MIN(DATE(f.created_at))                 AS first_purchase,
    MAX(DATE(f.created_at))                 AS last_purchase
FROM `ntupace.thelook_marts.fact_sales`    f
JOIN `ntupace.thelook_marts.dim_customer`  c ON f.customer_key = c.customer_key
WHERE f.status NOT IN ('cancelled','returned')
GROUP BY c.customer_key, c.first_name, c.last_name,
         c.age_bucket, c.gender, c.country, c.traffic_source
ORDER BY lifetime_value DESC
""")

print('CLV Distribution Summary')
print(clv[['total_orders','lifetime_value','avg_order_value']].describe().round(2))

fig, axes = plt.subplots(1, 2, figsize=(16, 5))

axes[0].hist(clv['lifetime_value'], bins=50, color='teal', edgecolor='white')
axes[0].set_title('Customer Lifetime Value Distribution', fontsize=13, fontweight='bold')
axes[0].set_xlabel('Lifetime Value ($)')
axes[0].set_ylabel('Customers')
axes[0].xaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'${x:,.0f}'))

clv_by_age = clv.groupby('age_bucket')['lifetime_value'].median().reset_index()
sns.barplot(data=clv_by_age, x='age_bucket', y='lifetime_value', palette='OrRd', ax=axes[1])
axes[1].set_title('Median CLV by Age Group', fontsize=13, fontweight='bold')
axes[1].set_xlabel('Age Group')
axes[1].set_ylabel('Median Lifetime Value ($)')
axes[1].yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'${x:,.0f}'))

plt.tight_layout()
plt.savefig('../reports/clv_analysis.png', dpi=150, bbox_inches='tight')
plt.show()

print('\nTop 10 Customers by Lifetime Value:')
clv[['full_name','country','age_bucket','total_orders','lifetime_value']].head(10)

## 6. Key Business KPIs

In [ ]:
kpis = q("""
SELECT
    COUNT(DISTINCT order_id)                              AS total_orders,
    COUNT(DISTINCT customer_key)                          AS unique_customers,
    ROUND(SUM(sale_price), 2)                             AS total_gmv,
    ROUND(SUM(gross_margin), 2)                           AS total_gross_profit,
    ROUND(AVG(gross_margin_pct) * 100, 1)                 AS avg_gross_margin_pct,
    ROUND(AVG(sale_price), 2)                             AS avg_item_price,
    ROUND(AVG(days_to_ship), 1)                           AS avg_days_to_ship,
    COUNTIF(status = 'returned') /
        COUNT(*) * 100                                    AS return_rate_pct
FROM `ntupace.thelook_marts.fact_sales`
WHERE status IN ('complete','returned','shipped')
""")

labels = {
    'total_orders':         ('Total Orders',          '{:,.0f}'),
    'unique_customers':     ('Unique Customers',       '{:,.0f}'),
    'total_gmv':            ('Total GMV',              '${:,.2f}'),
    'total_gross_profit':   ('Gross Profit',           '${:,.2f}'),
    'avg_gross_margin_pct': ('Avg Gross Margin',       '{:.1f}%'),
    'avg_item_price':       ('Avg Item Price',         '${:.2f}'),
    'avg_days_to_ship':     ('Avg Days to Ship',       '{:.1f} days'),
    'return_rate_pct':      ('Return Rate',            '{:.2f}%'),
}

print('=' * 55)
print('  THELOOK E-COMMERCE — KEY BUSINESS METRICS')
print('=' * 55)
for col, (label, fmt) in labels.items():
    val = kpis[col].iloc[0]
    print(f'  {label:<30}  {fmt.format(val):>12}')
print('=' * 55)